# 04 — Membangun API dan Web App NER

Jalankan setelah Notebook 03 berhasil menghasilkan file NER_Trained_Model_and_Report.zip. Notebook ini menyiapkan FastAPI, tampilan browser, tujuh tes API, dan demo melalui Colab.

Aplikasi harus menggunakan model kelompok yang dilatih/adaptasi, bukan diam-diam kembali ke model pretrained.

## Kontrak aplikasi

- POST /extract-entities menerima JSON berisi title dan body.
- Keduanya harus string dan tidak kosong.
- Batas awal: title 500 karakter, body 10.000 karakter.
- Entitas berisi text, label, start, end; meta berisi entity_count dan nama model.
- Posisi karakter dihitung terhadap teks gabungan title + satu newline + body.

In [ ]:
import io, json, zipfile, shutil, subprocess, sys, time, os
from pathlib import Path
from google.colab import files
from IPython.display import display, HTML
PROJECT = Path('/content/ner_service_project')
if PROJECT.exists(): shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)
print('Folder proyek:', PROJECT)

## 1. Unggah model hasil training dari Notebook 03

Unggah NER_Trained_Model_and_Report.zip yang dibuat oleh Notebook 03. Model harus ada pada file tersebut; bila tidak ada, kembali ke Notebook 03 dan selesaikan proses training terlebih dahulu.

In [ ]:
uploaded = files.upload()
zip_name = next((n for n in uploaded if n.lower().endswith('.zip')), None)
if not zip_name: raise ValueError('Unggah NER_Trained_Model_and_Report.zip dari Notebook 03.')
extract_dir = Path('/content/ner_model_package')
if extract_dir.exists(): shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as zf: zf.extractall(extract_dir)
model_src, meta_src = extract_dir/'model', extract_dir/'model_metadata.json'
if not (model_src/'config.cfg').exists(): raise FileNotFoundError('Folder model tidak ditemukan di ZIP.')
if not meta_src.exists(): raise FileNotFoundError('model_metadata.json tidak ditemukan di ZIP.')
(PROJECT/'models').mkdir(parents=True, exist_ok=True)
shutil.copytree(model_src, PROJECT/'models'/'ner_model')
shutil.copy2(meta_src, PROJECT/'models'/'model_metadata.json')
print('Model siap:', json.loads(meta_src.read_text(encoding='utf-8')))

## 2. Tulis file backend, frontend, tes, dan dependensi

In [ ]:
(PROJECT/'app').mkdir(parents=True, exist_ok=True)
(PROJECT/'app'/'__init__.py').write_text('', encoding='utf-8')
(PROJECT/'app'/'main.py').write_text("from pathlib import Path\nimport json, os\nimport spacy\nfrom fastapi import FastAPI, Request, HTTPException\nfrom fastapi.responses import HTMLResponse\n\nBASE_DIR = Path(__file__).resolve().parents[1]\nMODEL_DIR = Path(os.environ.get(\"NER_MODEL_PATH\", str(BASE_DIR / \"models\" / \"ner_model\")))\nMETA_PATH = BASE_DIR / \"models\" / \"model_metadata.json\"\nif not (MODEL_DIR / \"config.cfg\").exists():\n    raise RuntimeError(f\"Model terlatih tidak ditemukan di {MODEL_DIR}. Ikuti README.\")\nnlp = spacy.load(str(MODEL_DIR))\nnlp.max_length = 20000\nmetadata = json.loads(META_PATH.read_text(encoding=\"utf-8\")) if META_PATH.exists() else {}\nTARGET_LABELS = set(metadata.get(\"target_labels\", nlp.get_pipe(\"ner\").labels))\nMODEL_NAME = f\"{metadata.get('base_model', 'spaCy')} adapted on group annotations\"\nMAX_TITLE_CHARS, MAX_BODY_CHARS = 500, 10000\napp = FastAPI(title=\"Article Named-Entity Recognition Service\", version=\"1.0.0\")\n\n@app.get(\"/\", response_class=HTMLResponse)\ndef home():\n    return (BASE_DIR / \"web\" / \"index.html\").read_text(encoding=\"utf-8\")\n\n@app.get(\"/health\")\ndef health():\n    return {\"status\": \"ok\", \"model\": MODEL_NAME}\n\n@app.post(\"/extract-entities\")\nasync def extract_entities(request: Request):\n    try:\n        payload = await request.json()\n    except Exception:\n        raise HTTPException(status_code=400, detail=\"Body harus berupa JSON valid.\")\n    if not isinstance(payload, dict):\n        raise HTTPException(status_code=400, detail=\"JSON harus berupa object dengan title dan body.\")\n    title, body = payload.get(\"title\"), payload.get(\"body\")\n    if not isinstance(title, str) or not isinstance(body, str):\n        raise HTTPException(status_code=400, detail=\"title dan body wajib berupa string.\")\n    title, body = title.strip(), body.strip()\n    if not title:\n        raise HTTPException(status_code=400, detail=\"Judul artikel tidak boleh kosong.\")\n    if not body:\n        raise HTTPException(status_code=400, detail=\"Isi artikel tidak boleh kosong.\")\n    if len(title) > MAX_TITLE_CHARS:\n        raise HTTPException(status_code=400, detail=f\"Judul maksimal {MAX_TITLE_CHARS} karakter.\")\n    if len(body) > MAX_BODY_CHARS:\n        raise HTTPException(status_code=400, detail=f\"Isi artikel maksimal {MAX_BODY_CHARS} karakter.\")\n    analyzed_text = title + \"\\n\" + body\n    doc = nlp(analyzed_text)\n    entities = [\n        {\"text\": ent.text, \"label\": ent.label_, \"start\": ent.start_char, \"end\": ent.end_char}\n        for ent in doc.ents if ent.label_ in TARGET_LABELS\n    ]\n    return {\"entities\": entities, \"meta\": {\"entity_count\": len(entities), \"model\": MODEL_NAME}}\n", encoding='utf-8')
(PROJECT/'web').mkdir(parents=True, exist_ok=True)
(PROJECT/'web'/'index.html').write_text("<!doctype html>\n<html lang=\"id\"><head><meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width,initial-scale=1\">\n<title>Article NER Service</title>\n<style>\nbody{font-family:Arial,sans-serif;background:#f1f5f9;color:#172554;margin:0;padding:24px}.wrap{max-width:950px;margin:auto}.card{background:white;border:1px solid #dbeafe;border-radius:14px;padding:22px;margin:14px 0}h1{margin-bottom:6px}.muted{color:#64748b}label{display:block;font-weight:bold;margin:16px 0 8px}input,textarea{box-sizing:border-box;width:100%;padding:12px;border:1px solid #cbd5e1;border-radius:8px;font:inherit}textarea{min-height:180px}button{background:#2563eb;color:white;border:0;border-radius:8px;padding:12px 18px;margin-top:14px;font-weight:bold;cursor:pointer}button:disabled{opacity:.5}.status{display:none;padding:12px;border-radius:8px;margin-top:12px}.ok{display:block;background:#dcfce7;color:#166534}.err{display:block;background:#fee2e2;color:#991b1b}table{width:100%;border-collapse:collapse;margin-top:12px}td,th{text-align:left;padding:10px;border-bottom:1px solid #e2e8f0}th{background:#eff6ff}.tag{background:#dbeafe;color:#1d4ed8;padding:4px 8px;border-radius:20px;font-size:12px;font-weight:bold}.scroll{overflow:auto}\n</style></head><body><div class=\"wrap\">\n<div class=\"card\"><p class=\"muted\">NLP · Named Entity Recognition</p><h1>Article Named-Entity Recognition</h1><p class=\"muted\">Masukkan judul dan isi berita untuk menemukan nama orang, organisasi, dan tempat. Prediksi dapat mengandung kesalahan.</p></div>\n<div class=\"card\"><form id=\"form\"><label for=\"title\">Judul artikel</label><input id=\"title\" maxlength=\"500\" required placeholder=\"Judul artikel\"><label for=\"body\">Isi artikel</label><textarea id=\"body\" maxlength=\"10000\" required placeholder=\"Isi artikel\"></textarea><p class=\"muted\">Judul maksimal 500 karakter; isi maksimal 10.000 karakter.</p><button id=\"btn\">Analisis entitas</button></form><div id=\"status\" class=\"status\"></div></div>\n<div class=\"card\" id=\"result\" hidden><h2>Hasil entitas</h2><p class=\"muted\" id=\"summary\"></p><div class=\"scroll\"><table><thead><tr><th>Teks</th><th>Label</th><th>Start</th><th>End</th></tr></thead><tbody id=\"rows\"></tbody></table></div></div>\n<p class=\"muted\">API docs: <a href=\"/docs\" target=\"_blank\" rel=\"noopener\">/docs</a></p></div>\n<script>\nconst form=document.getElementById('form'),btn=document.getElementById('btn'),statusBox=document.getElementById('status'),result=document.getElementById('result'),summary=document.getElementById('summary'),rows=document.getElementById('rows');\nform.addEventListener('submit',async function(event){event.preventDefault();btn.disabled=true;btn.textContent='Menganalisis...';statusBox.className='status ok';statusBox.textContent='Sedang memproses artikel';statusBox.style.display='block';result.hidden=true;rows.replaceChildren();try{const response=await fetch('/extract-entities',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({title:document.getElementById('title').value,body:document.getElementById('body').value})});const data=await response.json();if(!response.ok){throw new Error(data.detail||'Permintaan gagal')}statusBox.textContent='Analisis selesai.';summary.textContent='Ditemukan '+data.meta.entity_count+' entitas. Model: '+data.meta.model;for(const ent of data.entities){const tr=document.createElement('tr');for(const value of [ent.text,ent.label,ent.start,ent.end]){const td=document.createElement('td');if(value===ent.label){const tag=document.createElement('span');tag.className='tag';tag.textContent=String(value);td.appendChild(tag)}else{td.textContent=String(value)}tr.appendChild(td)}rows.appendChild(tr)}if(!data.entities.length){const tr=document.createElement('tr'),td=document.createElement('td');td.colSpan=4;td.textContent='Tidak ada entitas yang terdeteksi.';tr.appendChild(td);rows.appendChild(tr)}result.hidden=false}catch(error){statusBox.className='status err';statusBox.textContent='Gagal: '+error.message}finally{btn.disabled=false;btn.textContent='Analisis entitas'}});\n</script></body></html>", encoding='utf-8')
(PROJECT/'tests').mkdir(parents=True, exist_ok=True)
(PROJECT/'tests'/'test_api.py').write_text("from fastapi.testclient import TestClient\nfrom app.main import app\nclient = TestClient(app)\n\ndef test_valid_schema():\n    r = client.post(\"/extract-entities\", json={\"title\":\"Sample headline\",\"body\":\"Ada Lovelace visited London.\"})\n    assert r.status_code == 200\n    data = r.json()\n    assert isinstance(data[\"entities\"], list)\n    assert data[\"meta\"][\"entity_count\"] == len(data[\"entities\"])\n    for item in data[\"entities\"]:\n        assert all(k in item for k in (\"text\",\"label\",\"start\",\"end\"))\n        assert item[\"start\"] < item[\"end\"]\n\ndef test_empty_title():\n    assert client.post(\"/extract-entities\", json={\"title\":\" \",\"body\":\"Some article.\"}).status_code == 400\n\ndef test_empty_body():\n    assert client.post(\"/extract-entities\", json={\"title\":\"Headline\",\"body\":\" \"}).status_code == 400\n\ndef test_non_string():\n    assert client.post(\"/extract-entities\", json={\"title\":123,\"body\":\"Some article.\"}).status_code == 400\n\ndef test_long_body():\n    assert client.post(\"/extract-entities\", json={\"title\":\"Headline\",\"body\":\"x\"*10001}).status_code == 400\n\ndef test_bad_json():\n    r = client.post(\"/extract-entities\", content=\"{bad json\", headers={\"Content-Type\":\"application/json\"})\n    assert r.status_code == 400\n\ndef test_deterministic():\n    payload = {\"title\":\"Sample headline\",\"body\":\"Ada Lovelace visited London.\"}\n    assert client.post(\"/extract-entities\", json=payload).json() == client.post(\"/extract-entities\", json=payload).json()\n", encoding='utf-8')
requirements = 'fastapi>=0.110,<1.0\\nuvicorn[standard]>=0.27,<1.0\\nspacy>=3.7,<4.0\\npytest>=8,<9\\nhttpx>=0.27\\n'
(PROJECT/'requirements.txt').write_text(requirements.replace('\\\\n','\\n'), encoding='utf-8')
readme = '''# Article Named-Entity Recognition Service

## Jalankan lokal
1. Pastikan folder models/ner_model berisi model terlatih dan models/model_metadata.json tersedia.
2. Gunakan Python 3.10 atau lebih baru.
3. Jalankan pip install -r requirements.txt
4. Jalankan uvicorn app.main:app --host 127.0.0.1 --port 8000 dari folder proyek.
5. Buka http://127.0.0.1:8000 atau dokumentasi API http://127.0.0.1:8000/docs.
6. Jalankan tes dengan pytest -q.

Endpoint POST /extract-entities menerima object JSON dengan title dan body. Batas title 500 karakter dan body 10.000 karakter. Offset start/end dihitung terhadap teks gabungan title, newline, dan body.

Model dalam aplikasi harus sama dengan model yang dicatat pada laporan evaluasi. NER masih dapat salah mengenali atau melewatkan entitas.
'''
(PROJECT/'README.md').write_text(readme, encoding='utf-8')
print('Backend, HTML, tes, requirements, dan README dibuat.')

## 3. Instal dependensi dan jalankan tes otomatis

In [ ]:
%pip -q install -r /content/ner_service_project/requirements.txt

In [ ]:
import os, sys
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
!pytest -q /content/ner_service_project/tests/test_api.py

## 4. Jalankan server dan buka web app di Colab

In [ ]:
import requests
log_file = open('/content/ner_uvicorn.log','w')
server = subprocess.Popen([sys.executable,'-m','uvicorn','app.main:app','--host','0.0.0.0','--port','8000'], cwd=str(PROJECT), stdout=log_file, stderr=subprocess.STDOUT)
ready = False
for _ in range(30):
    try:
        res = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if res.ok:
            print('Server aktif:', res.json()); ready=True; break
    except Exception:
        time.sleep(1)
if not ready:
    print(Path('/content/ner_uvicorn.log').read_text(encoding='utf-8')[-4000:])
    raise RuntimeError('Server belum aktif; periksa log di atas.')
from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(8000)')
display(HTML('<p><a target="_blank" href="'+url+'">Buka aplikasi Article NER</a></p><p><a target="_blank" href="'+url+'/docs">Buka dokumentasi API</a></p>'))

## 5. Ekspor proyek

ZIP diunduh agar struktur folder, model, aplikasi, test, dan README bisa dipindahkan untuk dijalankan lokal. Jangan unggah dataset mentah ke repository publik. Model besar boleh disimpan sebagai artefak terpisah sesuai aturan dosen.

In [ ]:
archive_path = shutil.make_archive('/content/Article_NER_Service_Project','zip',PROJECT)
print('Paket dibuat:', archive_path)
files.download(archive_path)
print('Ambil screenshot form, hasil analisis, error validasi, hasil pytest, dan tabel metrik dari Notebook 03.')

## Checklist

- [ ] Model web sama dengan model yang dievaluasi.
- [ ] Request valid menghasilkan JSON dengan entities dan meta.
- [ ] Input kosong, tipe salah, JSON rusak, dan teks terlalu panjang ditolak.
- [ ] Minimal lima tes otomatis lulus.
- [ ] README menjelaskan instalasi, startup, endpoint, model, contoh request/response, dan keterbatasan.
- [ ] Laporan mencakup exact-span precision, recall, F1, metrik per label, baseline vs model terlatih, serta minimal tiga kesalahan yang nyata.